# Dapr - JupyterLab

Dieses Hands-on zeigt, wie die OpenAI API als Dapr-Komponente konfiguriert wird und wie anschliessend über ein mit Sidecar gestartetes Jupyter Lab eine Kommunikation mit OpenAI über Dapr erfolgt.

Verbindung zum LLM einrichten

* [OpenAI](https://docs.dapr.io/reference/components-reference/supported-conversation/openai/)

Endpunkte
* openai
* llm-provider


In [ ]:
%run ~/work/env.py
! cat ~/work/env.py

In [ ]:
%%bash
source ~/work/env.py
kubectl apply -f - <<EOF
apiVersion: dapr.io/v1alpha1
kind: Component
metadata:
  name: openai
spec:
  type: conversation.openai
  version: v1
  metadata:
  - name: key
    value: "${OPENAI_API_KEY}"
  - name: model
    value: "gpt-5.4"
  - name: endpoint
    value: "https://api.openai.com/v1"
  - name: responseCacheTTL
    value: "10m"
EOF

In [ ]:
%%bash
source ~/work/env.py
kubectl apply -f - <<EOF
apiVersion: dapr.io/v1alpha1
kind: Component
metadata:
  name: llm-provider
spec:
  type: conversation.openai
  version: v1
  metadata:
  - name: key
    value: "${OPENAI_API_KEY}"
  - name: model
    value: "${AI_MODEL}"
  - name: endpoint
    value: "${AI_BASE_URL}"
  - name: responseCacheTTL
    value: "10m"
EOF

Agent Store

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: dapr.io/v1alpha1
kind: Component
metadata:
  name: agent-workflow
spec:
  type: state.redis
  version: v1
  metadata:
    - name: redisHost
      value: dapr-dev-redis-master.default.svc.cluster.local:6379
    - name: redisPassword
      secretKeyRef:
        name: dapr-dev-redis
        key: redis-password
    - name: actorStateStore
      value: "true"
---
apiVersion: dapr.io/v1alpha1
kind: Component
metadata:
  name: agent-memory
spec:
  type: state.redis
  version: v1
  metadata:
    - name: redisHost
      value: dapr-dev-redis-master.default.svc.cluster.local:6379
    - name: redisPassword
      secretKeyRef:
        name: dapr-dev-redis
        key: redis-password
EOF

## Jupyter Lab

Die Lab-Umgebung wird zusammen mit einem Dapr-Sidecar gestartet und ermöglicht so den Zugriff auf das LLM über die zuvor konfigurierte Dapr-Komponente.


In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: jupyter-dapr
  namespace: default
spec:
  replicas: 1
  selector:
    matchLabels:
      app: jupyter-dapr
  template:
    metadata:
      labels:
        app: jupyter-dapr
      annotations:
        dapr.io/enabled: "true"
        dapr.io/app-id: "jupyter-dapr"
        dapr.io/app-port: "8888"
        dapr.io/log-level: "debug"
    spec:
      containers:
      - name: notebook
        image: quay.io/jupyter/minimal-notebook:latest
        ports:
        - containerPort: 8888
        env:
        - name: JUPYTER_TOKEN
          value: "changeme"
        command:
          - start-notebook.sh
          - --NotebookApp.token=''
          - --NotebookApp.password=''
          - --NotebookApp.allow_origin='*'
          - --NotebookApp.ip=0.0.0.0
          - --NotebookApp.allow_root=true
        volumeMounts:
        - name: data-volume
          mountPath: /home/jovyan/data  # im Juypter Lab
      volumes:
      - name: data-volume
        hostPath:
          path: /home/ubuntu/cna-ai/2_Unterrichtsressourcen/08-agents/lab   # entspricht $HOME/cna-ai/... auf dem Node
          type: DirectoryOrCreate
---
apiVersion: v1
kind: Service
metadata:
  name: jupyter-dapr
  namespace: default
spec:
  selector:
    app: jupyter-dapr
  ports:
  - name: http
    port: 8888
    targetPort: 8888
  type: NodePort
EOF


### JupyterLab-Deployment

Dieses JupyterLab-Deployment wurde bewusst mit einem Dapr Sidecar aufgebaut, um eine experimentelle und zugleich produktionsnahe Umgebung für verteilte, KI-gestützte Anwendungen bereitzustellen.

Mittels folgenden URLs kann auf das Jupyter Lab und weitere Dashboard zugegriffen werden:

In [ ]:
%%bash
echo "Dapr Jupyter Lab: http://"$(cat ~/work/server-ip)":$(kubectl get service jupyter-dapr -o=jsonpath='{ .spec.ports[0].nodePort }')/"
echo "K8s Dashboard   : https://$(cat ~/work/server-ip):30443"
echo "Dapr Dashboard  : http://"$(cat ~/work/server-ip)":$(kubectl get -n dapr-system service dapr-dashboard -o=jsonpath='{ .spec.ports[0].nodePort }')/"

---
### Aufräumen


In [ ]:
%%bash
kubectl delete component openai || true
kubectl delete component llm-provider || true
kubectl delete deployment jupyter-dapr || true
kubectl delete service jupyter-dapr || true
kubectl delete component agent-workflow || true
kubectl delete component agent-memory || true
